advanced agentic rag (filesystem based)

In [ ]:
import os
import re
from typing import Any, Dict

import mlflow
from rich import print as rptint
from dotenv import load_dotenv
import numpy as np
from pymongo import MongoClient

from langchain.agents import create_agent
from langchain_core.documents import Document
from langchain.agents.middleware import AgentMiddleware, AgentState
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_mongodb import MongoDBAtlasVectorSearch
from langchain_mongodb.retrievers.hybrid_search import MongoDBAtlasHybridSearchRetriever

mlflow.langchain.autolog()
load_dotenv()

MONGODB_URI = os.getenv('MONGODB_URI')
DB_NAME = 'agent_db'
COLLECTION_NAME = 'cxi-travel-ins-collection'
ATLAS_VECTOR_SEARCH_INDEX_NAME = "cxi-travel-ins-qa-vector-index"

client = MongoClient(MONGODB_URI)
MONGODB_COLLECTION = client[DB_NAME][COLLECTION_NAME]

In [ ]:
client = MongoClient(MONGODB_URI)
embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001')
llm = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash'
)
vector_store = MongoDBAtlasVectorSearch(
    collection=MONGODB_COLLECTION,
    embedding=embeddings,
    index_name=ATLAS_VECTOR_SEARCH_INDEX_NAME,
    relevance_score_fn="cosine",
    embedding_key="embedding",
    text_key="text"
)
retriever = MongoDBAtlasHybridSearchRetriever(
    vectorstore=vector_store,
    search_index_name="search_index",
    top_k=5,
    fulltext_penalty=50,
    vector_penalty=50,
    post_filter=[
        {
            '$project': {
                'embedding': 0,
            }
        }
    ]
)

In [ ]:
class State(AgentState):
    context: list[Document]
    
class HybridRetrieveDocumentsMiddleware(AgentMiddleware[State]):
    state_schema = State
    
    def before_model(self, state: AgentState) -> dict[str, Any] | None:
        last_message = state['messages'][-1]
        retrieved_docs = retriever.invoke(last_message.content)
        
        docs = "\n\n".join([doc.page_content for doc in retrieved_docs])
        
        augmented_context = (
            f"{last_message.content}\n\n"
            f"retrevel result: {docs}\n\n"
            "Use the above result to answer the user's question."
            "If you don't know the answer, just say 'I don't know'."
            "使用繁體中文回答"
        )
        
        return {
            "messages": [last_message.model_copy(update={"content": augmented_context})],
            "context": retrieved_docs,
        }

In [ ]:
rag_llm = create_agent(
    model=llm,
    tools=[],
    middleware=[HybridRetrieveDocumentsMiddleware()]
)

In [ ]:
rag_llm.invoke({"messages": [{"role": "user", "content": "行李遺失後應該如何申請理賠？"}]})